# Submission final NLI con CatBoost

Este cuaderno vuelve a entrenar CatBoost con todos los pares etiquetados y predice el test con la configuración evaluada en el holdout traducido. El texto ya se tradujo y preprocesó una vez en el equipo. Se adjunta como dataset de Kaggle la carpeta generada por `prepare_kaggle_delivery.py`, con datos procesados, manifiesto y código fuente del pipeline. El cuaderno genera `submission.csv` dentro de la ejecución de Kaggle.

Las características TF-IDF, el contraste entre premisa e hipótesis, los ratios POS y las señales de relación usan las clases existentes del proyecto. El vocabulario y la selección chi-cuadrado se ajustan aquí con el train completo. Las etiquetas de test no están en los artefactos de entrada.

**Preparación local**

```powershell
.venv/Scripts/python.exe prepare_kaggle_delivery.py
```

Sube `artifacts/kaggle_delivery/` como dataset privado de Kaggle y adjúntalo al cuaderno. El ajuste y la predicción se ejecutan en el entorno donde se guarda la versión del cuaderno.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata as package_metadata
import json
import os
import sys
import tempfile
import time
import zipfile

import pandas as pd

ON_KAGGLE = Path("/kaggle/input").is_dir()
if ON_KAGGLE:
    configured_data_dir = os.environ.get("NLI_DELIVERY_DIR")
    if configured_data_dir is None:
        candidates = list(Path("/kaggle/input").glob("*/training_source.zip"))
        if len(candidates) != 1:
            raise RuntimeError("Adjunta una sola carpeta kaggle_delivery o define NLI_DELIVERY_DIR.")
        configured_data_dir = str(candidates[0].parent)
    DATA_DIR = Path(configured_data_dir)
    OUTPUT_DIR = Path(os.environ.get("NLI_OUTPUT_DIR", "/kaggle/working"))
else:
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "training.py").is_file():
        PROJECT_DIR = PROJECT_DIR / "0-AiDea"
    DATA_DIR = Path(os.environ.get("NLI_DELIVERY_DIR", str(PROJECT_DIR / "artifacts/kaggle_delivery")))
    OUTPUT_DIR = Path(os.environ.get("NLI_OUTPUT_DIR", str(PROJECT_DIR / "results_kaggle_translated")))

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
RUN_MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"
if SUBMISSION_PATH.exists() or RUN_MANIFEST_PATH.exists():
    raise FileExistsError(f"Se conserva la submission o manifiesto existente en {OUTPUT_DIR}")
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
assert manifest["schema_version"] == 1
for filename, expected_hash in manifest["artifacts"].items():
    actual_hash = hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"El hash de {filename} no coincide"

source_dir = Path(tempfile.mkdtemp(prefix="nli_training_source_"))
with zipfile.ZipFile(DATA_DIR / "training_source.zip") as source_archive:
    source_archive.extractall(source_dir)
for filename, expected_hash in manifest["training_source_sha256"].items():
    source_hash = hashlib.sha256((source_dir / filename).read_bytes()).hexdigest()
    assert source_hash == expected_hash, f"El módulo {filename} no coincide con el manifest"
sys.path.insert(0, str(source_dir))

from training import LABEL_NAMES, build_pipeline, write_submission

print("Entrenamiento:", manifest["train_rows"], "filas")
print("Predicción:", manifest["test_rows"], "filas")
print("Traducción:", manifest["translation_model"])

C:\Users\javiy\Documents\training\0-AiDea\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Entrenamiento: 12120 filas
Predicción: 5195 filas
Traducción: gemma3:4b


In [2]:
train = pd.read_json(DATA_DIR / "train_processed.jsonl", lines=True, dtype={"id": str})
test = pd.read_json(DATA_DIR / "test_processed.jsonl", lines=True, dtype={"id": str})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})

train_required = {"id", "label", "lang_abv", "premise_processed", "hypothesis_processed"}
test_required = {"id", "lang_abv", "premise_processed", "hypothesis_processed"}
assert train_required.issubset(train.columns)
assert test_required.issubset(test.columns)
assert "label" not in test.columns
assert train.id.is_unique and test.id.is_unique
assert train.label.notna().all()
assert set(train.label.astype(int)) == set(LABEL_NAMES)
assert sample.columns.tolist() == ["id", "prediction"]
assert sample.id.tolist() == test.id.tolist()
actual_id_hash = hashlib.sha256("\n".join(test.id.astype(str)).encode("utf-8")).hexdigest()
assert actual_id_hash == manifest["test_ids_sha256"]

print("Distribución de etiquetas de train:", train.label.value_counts().sort_index().to_dict())
print("IDs de test coinciden, en orden, con sample_submission.")

Distribución de etiquetas de train: {0: 4176, 1: 3880, 2: 4064}
IDs de test coinciden, en orden, con sample_submission.


## Entrenamiento final

La búsqueda de hiperparámetros ya se hizo en los datos de desarrollo. Se conserva el CatBoost con 300 iteraciones, profundidad 6 y tasa de aprendizaje 0,1. El ajuste final usa las 12 120 etiquetas disponibles. La selección de 300 variables se ajusta con ese train. El test solo se usa para predecir.

In [3]:
SEED = 42
VECTOR_METHOD = {
    "method": "pair_diff",
    "columns": ("premise_processed", "hypothesis_processed"),
    "min_df": 5,
    "word_ngram_range": (1, 1),
    "use_pos": False,
}
CATBOOST_PARAMS = {
    "iterations": 300,
    "learning_rate": 0.1,
    "depth": 6,
    "loss_function": "MultiClass",
    "verbose": False,
    "allow_writing_files": False,
    "thread_count": -1,
}

pipeline = build_pipeline(
    vectorizer_params=VECTOR_METHOD,
    classifier="catboost",
    classifier_params=CATBOOST_PARAMS,
    use_pos_features=True,
    use_relation_features=True,
    select_k=300,
    random_state=SEED,
)
started = time.perf_counter()
pipeline.fit(train, train.label.astype("int64"))
fit_seconds = round(time.perf_counter() - started, 2)
print(f"Tiempo de ajuste final: {fit_seconds} s")

Tiempo de ajuste final: 6.03 s


In [4]:
prediction_values = [int(value) for value in pipeline.predict(test)]
assert len(prediction_values) == len(test)
assert set(prediction_values).issubset(set(LABEL_NAMES))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission = write_submission(test.id, prediction_values, SUBMISSION_PATH)
assert submission.columns.tolist() == sample.columns.tolist()
assert submission.id.tolist() == sample.id.tolist()

run = {
    "classifier": "catboost",
    "classifier_params": CATBOOST_PARAMS,
    "vectorizer": {**VECTOR_METHOD, "columns": list(VECTOR_METHOD["columns"])},
    "seed": SEED,
    "select_k": 300,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "test_ids_sha256": actual_id_hash,
    "translation_model": manifest["translation_model"],
    "preprocessor_params": manifest["preprocessor_params"],
    "fit_seconds": fit_seconds,
    "submission_sha256": hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest(),
    "versions": {name: package_metadata.version(name) for name in ("pandas", "scikit-learn", "catboost")},
}
RUN_MANIFEST_PATH.write_text(
    json.dumps(run, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Submission creada:", SUBMISSION_PATH)
print("Predicciones por clase:", submission.prediction.value_counts().sort_index().to_dict())

Submission creada: C:\Users\javiy\Documents\training\0-AiDea\results_kaggle_translated\submission.csv
Predicciones por clase: {0: 1927, 1: 1909, 2: 1359}


## Entrega

En Kaggle, ejecuta y guarda una versión de este cuaderno. La submission la genera el propio cuaderno en `/kaggle/working/submission.csv`. Guarda también `run_manifest.json`, que registra semilla, parámetros, versiones, duración e integridad de los IDs. El resultado es una salida local del cuaderno.